# Pipeline v5 (z705) — par-nivel, escalado de la cátedra como *offset*, sin fugas

Sucesor del `z702`. Mantiene el esquema de la pizarra (densa cátedra → escalado + FE →
clase t+2 → cluster DTW → Optuna/LightGBM Tweedie → predict desescalado → submit) y
corrige lo que estaba mal. Los números que se citan están medidos sobre el dataset real
(ver `z705_plan_mejoras_v3.md`).

**Ley inviolable.** La clase es lo único que mira el futuro (`tn(t+2)`). Todas las
features salen de `<= t`. La celda 5 lo verifica *empíricamente*: recalcula el FE sobre
la historia truncada en T y exige que las features de las filas `periodo <= T` no
cambien. Si algo mira adelante, ese test lo caza y el notebook se corta.

**Qué cambia respecto de z702**

1. **Bug de fuga**: en `rmean_3 / rmean_12 / rstd_3` el `.over(agrupa_id)` estaba dentro
   del `shift`, así que la ventana móvil corría sobre la columna entera y se derramaba
   de la cola de la serie anterior — o sea, arrastraba periodos *posteriores* de otro
   par. Corregido y cubierto por el test de causalidad.
2. **Validación con el horizonte del deploy**: se entrena hasta `ancla − 2`, no hasta
   `ancla − 1`. Antes la validación tenía un mes más de información del que hay en
   producción.
3. **WAPE global**: la métrica se calcula una sola vez sobre las predicciones
   ensambladas, no por cluster sobre sumas parciales de cada producto.
4. **Escalado como offset**: se conserva `s(t)` = media expandida de la cátedra, pero en
   lugar de dividir el target se pasa `init_score = log(s)` con Tweedie (link log). La
   reconstrucción `pred = s·exp(f)` es el mismo desescalado multiplicativo de siempre.
   Lo que cambia: la pérdida queda en toneladas — que es como pesa el WAPE, y el 1 % de
   las filas concentra el 74,6 % del tonelaje — y las filas con `s = 0` (46 % del
   dataset, 43,6 % de los pares en inferencia) dejan de tener el target aplastado a cero
   y la predicción clavada en cero.
5. **Multiplicador estimado**: mediana ponderada de `y_p/ŷ_p` en validación (que es el
   argmin exacto del numerador del WAPE), con diagnóstico por decil y encogimiento hacia
   1. No más 0.9 / 1.0 / 1.1 a ciegas ni elegido por leaderboard público.
6. **Clustering DTW a nivel producto**, con ventana común y banda honesta, recalculado
   con corte causal para cada fold.

In [ ]:
# ruff: noqa: E402
%pip install -q optuna dtaidistance lightgbm polars duckdb pandas numpy scipy scikit-learn
# import os; os.environ['LABO3_BUCKET'] = '/ruta/al/bucket'

In [ ]:
import gc
import json
import os
import subprocess
import time
from pathlib import Path

import duckdb
import lightgbm as lgb
import numpy as np
import pandas as pd
import polars as pl
from scipy.cluster.hierarchy import fcluster, linkage
from scipy.optimize import minimize
from scipy.spatial.distance import squareform
from sklearn.metrics import silhouette_score

try:
    import optuna
    optuna.logging.set_verbosity(optuna.logging.WARNING)
    _HAY_OPTUNA = True
except Exception:
    _HAY_OPTUNA = False

try:
    from dtaidistance import dtw
    _C_OK = dtw.try_import_c()
except Exception:
    dtw, _C_OK = None, False

N_CORES = os.cpu_count() or 8


def resolver_bucket() -> Path:
    """Encuentra el bucket: variable de entorno, montaje GCS/Colab, o ./bucket local."""
    env = os.environ.get("LABO3_BUCKET")
    if env:
        p = Path(env); p.mkdir(parents=True, exist_ok=True); return p
    for c in ("/home/jupyter/buckets/b1", "/content/buckets/b1", "~/buckets/b1"):
        p = Path(c).expanduser()
        if p.exists():
            return p
    p = Path.cwd() / "bucket"; p.mkdir(parents=True, exist_ok=True); return p


def escribir_atomico(fn, destino: Path):
    """Escribe a un temporal y renombra: un corte nunca deja un archivo a medias."""
    destino = Path(destino); destino.parent.mkdir(parents=True, exist_ok=True)
    tmp = destino.with_suffix(destino.suffix + ".tmp")
    fn(str(tmp)); os.replace(tmp, destino)


def leer_json(p):
    return json.loads(Path(p).read_text())


def periodo_mas_h(p: int, h: int) -> int:
    y, m = divmod(int(p), 100); tot = y * 12 + (m - 1) + h
    return (tot // 12) * 100 + (tot % 12) + 1


print(f"cores={N_CORES}  optuna={_HAY_OPTUNA}  dtw_C={_C_OK}")

## 1) Parámetros

In [ ]:
PARAM = {
    "modo_test": False,              # True = smoke test (pocos productos, pocos trials)

    "horizonte": 2,
    "periodo_inferencia": 201912,    # parados aca -> predecir 202002
    "solo_productos_target": False,  # entrena con los 1233, predice los 780

    # ---- escalado ----
    # 'offset'  : init_score = log(s) con Tweedie  -> pred = s * exp(f)   [recomendado]
    # 'dividir' : target = tn(t+2)/s               -> pred = s * yhat     [z702, para A/B]
    "modo_escala": "offset",
    "piso_escala": 1e-3,             # 1 kg: evita log(0). Con offset el piso NO divide,
                                     # solo corre el punto de arranque del boosting.

    # ---- features (todas causales: <= t) ----
    "features_base": [
        "lag_1", "lag_2", "lag_3", "lag_12",
        "rmean_3", "rmean_12", "rstd_3",
        "nivel_relativo", "tendencia_3_12",
        "meses_consec_sin_compra", "ventas_ult12",
        "mes", "share_prod_en_cat3", "cat3", "brand",
    ],
    # el modelo par-nivel nunca ve el agregado que la metrica mide: se lo damos
    "usar_features_agregado": True,
    "features_agregado": ["tn_prod", "tn_cli", "share_par_en_prod",
                          "tend_prod_3_12", "nivel_prod_pos"],
    "usar_cluster_como_feature": True,
    "cols_categoricas": ["cat3", "brand", "cluster_id"],
    "eps": 1e-6,

    # ---- submuestreo de pares dormidos (sin compras en 12 meses) ----
    # son el 52 % de las filas y el 2,7 % del tonelaje. Se muestrean y se compensa con
    # peso 1/tasa: es el mismo estimador de la perdida, no se tira informacion.
    "submuestreo_dormidos": 0.25,

    # ---- clustering DTW ----
    "cl_nivel": "producto",          # 'producto' (1233 series, matriz completa) | 'ninguno'
    "cl_ventana": 24,                # meses hasta el corte: largo comun -> banda honesta
    "cl_banda": 3,                   # Sakoe-Chiba fijo
    "cl_lista_k": [4, 5, 6, 7, 8],
    "cl_min_balance": 0.05,
    "cl_linkage": "average",

    # ---- validacion ----
    "anclas_val": [201712, 201812],  # t+2 cae en febrero (el target es febrero)
    "anclas_control": [201906],      # ancla no-febrero, de control
    "peso_febrero": 2.0,

    # ---- LightGBM / Tweedie ----
    "max_bin": 1023,
    "num_threads": N_CORES,
    "tweedie_vp_rango": (1.1, 1.6),
    "techo_arboles": 900,
    "n_trials": 25,
    "semillas_ensemble": [102191, 314159, 777773],

    # ---- multiplicador ----
    "lambda_shrink": 0.7,            # m_final = 1 + lambda*(m* - 1)
    "probar_calibracion_afin": True,
    "margen_afin": 0.02,             # la afin tiene que ganar por 2% para usarse

    # ---- submit ----
    "clip_min": 0.0,
    "kaggle_competition": "labo-iii-2026-ba",
    "submit": True,

    "semilla": 102191,
}

if PARAM["modo_test"]:
    PARAM.update({
        "n_productos_test": 40,
        "cl_lista_k": [2, 3],
        "n_trials": 4,
        "techo_arboles": 300,
        "semillas_ensemble": [102191],
        "anclas_control": [],
        "submit": False,
    })

H = PARAM["horizonte"]
PISO = PARAM["piso_escala"]
BUCKET = resolver_bucket()
DIR_RAW = BUCKET / "datasets"
DIR_OUT = BUCKET / ("pipe_v5_test" if PARAM["modo_test"] else "pipe_v5")
DIR_RAW.mkdir(parents=True, exist_ok=True)
DIR_OUT.mkdir(parents=True, exist_ok=True)
print(f"BUCKET={BUCKET}\nOUT={DIR_OUT}\nescala={PARAM['modo_escala']}")

## 2) Datos crudos

In [ ]:
def descargar(archivo):
    dst = DIR_RAW / archivo
    if dst.exists():
        return
    url = f"https://storage.googleapis.com/open-courses/austral2026-5da5/labo3/{archivo}"
    subprocess.run(["wget", "-q", url, "-O", str(dst)], check=True)


for _a in ("sell-in.txt.gz", "tb_productos.txt", "product_id_apredecir201912.txt"):
    descargar(_a)

PROD_TARGET = set(pl.read_csv(DIR_RAW / "product_id_apredecir201912.txt",
                              separator="\t")["product_id"].cast(pl.Int64).to_list())
print("productos a predecir:", len(PROD_TARGET))

## 3) Densa zero-fill (criterio cátedra z601)

Grid ⟨cliente, producto, periodo⟩ con `tn = 0` donde no hubo venta, dentro de la vida del
producto (forzada hasta 201912 para los 780) y a partir del primer periodo del cliente.
Los meses quedan contiguos, así que `shift(k)` es exactamente el lag de k meses — y hay
un assert que lo verifica en vez de darlo por sentado.

In [ ]:
PATH_PRE = DIR_OUT / "preprocesado.parquet"


def _construir_preprocesado():
    con = duckdb.connect(); con.execute("SET preserve_insertion_order=false")
    con.execute(f"""CREATE TABLE sellin AS
        SELECT CAST(customer_id AS INT) customer_id, CAST(product_id AS INT) product_id,
               CAST(periodo AS INT) periodo, CAST(tn AS DOUBLE) tn
        FROM read_csv_auto('{DIR_RAW / 'sell-in.txt.gz'}')""")
    con.execute(f"""CREATE TABLE apredecir AS SELECT CAST(product_id AS INT) product_id
        FROM read_csv_auto('{DIR_RAW / 'product_id_apredecir201912.txt'}')""")

    filtro = ""
    if PARAM["solo_productos_target"]:
        filtro = "WHERE product_id IN (SELECT product_id FROM apredecir)"
    elif PARAM["modo_test"]:
        filtro = (f"WHERE product_id IN (SELECT product_id FROM apredecir "
                  f"ORDER BY product_id LIMIT {PARAM['n_productos_test']})")

    con.execute(f"""CREATE TABLE base AS
        SELECT customer_id, product_id, periodo, SUM(tn) tn FROM sellin {filtro}
        GROUP BY 1, 2, 3""")
    con.execute("CREATE TABLE periodos AS SELECT DISTINCT periodo FROM base")
    con.execute("""CREATE TABLE vida_prod AS
        SELECT product_id, MIN(periodo) nace, MAX(periodo) muere FROM base GROUP BY 1""")
    con.execute("""UPDATE vida_prod SET muere = 201912
        WHERE product_id IN (SELECT product_id FROM apredecir) AND muere < 201912""")
    con.execute("""CREATE TABLE primer_cli AS
        SELECT customer_id, MIN(periodo) nace_cli FROM base GROUP BY 1""")
    con.execute("""CREATE TABLE densa AS
        SELECT g.customer_id, g.product_id, g.periodo, COALESCE(b.tn, 0.0) tn
        FROM (SELECT c.customer_id, v.product_id, p.periodo
              FROM vida_prod v JOIN periodos p ON p.periodo BETWEEN v.nace AND v.muere
              CROSS JOIN primer_cli c WHERE p.periodo >= c.nace_cli) g
        LEFT JOIN base b USING (customer_id, product_id, periodo)""")

    prods = con.execute(f"""SELECT CAST(product_id AS INT) product_id, cat3, brand
        FROM read_csv_auto('{DIR_RAW / 'tb_productos.txt'}')""").df()
    df = con.execute("SELECT * FROM densa ORDER BY customer_id, product_id, periodo").pl()
    con.close()

    df = df.join(pl.from_pandas(prods).unique(subset=["product_id"]), on="product_id", how="left")
    MULT = 100_000
    return df.with_columns(
        (pl.col("customer_id").cast(pl.Int64) * MULT + pl.col("product_id").cast(pl.Int64)).alias("agrupa_id"))


if PATH_PRE.exists():
    print("[preprocesado] RESUME"); df_pre = pl.read_parquet(PATH_PRE)
else:
    df_pre = _construir_preprocesado()
    escribir_atomico(lambda t: df_pre.write_parquet(t), PATH_PRE)

_d = (df_pre.sort(["agrupa_id", "periodo"])
      .with_columns(pl.col("periodo").shift(1).over("agrupa_id").alias("_ant"))
      .drop_nulls("_ant")
      .with_columns((pl.col("periodo") - pl.col("_ant")).alias("_d")))
_saltos = _d.filter(~pl.col("_d").is_in([1, 89])).height       # 89 = diciembre -> enero
assert _saltos == 0, f"hay {_saltos} saltos de mes dentro de una serie: shift(k) != lag k"
print("preprocesado:", df_pre.shape, "| series:", df_pre["agrupa_id"].n_unique(), "| meses contiguos OK")

## 4) Escalado + Feature Engineering (causal)

`s(t)` = media expandida de `tn` hasta t inclusive: la escala de la cátedra, intacta.
Con `modo_escala='offset'` **no se divide nada**: `s` viaja como `init_score = log(s)` y
el desescalado del predict es `pred = s·exp(f)`. La clase es `tn(t+2)` en toneladas.

Las features de agregado (`tn_prod`, `tn_cli`, …) usan el periodo t de *otros* pares:
eso es presente, no futuro — parados en 201912 conocemos todo 201912.

In [ ]:
PATH_FE = DIR_OUT / "features.parquet"


def _fe(df: pl.DataFrame) -> pl.DataFrame:
    g = "agrupa_id"; eps = PARAM["eps"]
    df = df.sort([g, "periodo"])

    # escala catedra: media expandida hasta t (incluye t)
    df = df.with_columns(
        pl.col("tn").cum_sum().over(g).alias("_cs"),
        (pl.int_range(pl.len()).over(g) + 1).alias("_n"),
        pl.int_range(pl.len()).over(g).alias("_rn"),
    ).with_columns((pl.col("_cs") / pl.col("_n")).alias("s"))

    # OJO: el .over(g) va al FINAL de la cadena. Si se pone dentro (sobre el shift), el
    # rolling corre sobre la columna entera y se derrama entre series.
    df = df.with_columns([
        pl.col("tn").shift(1).over(g).alias("lag_1"),
        pl.col("tn").shift(2).over(g).alias("lag_2"),
        pl.col("tn").shift(3).over(g).alias("lag_3"),
        pl.col("tn").shift(12).over(g).alias("lag_12"),
        pl.col("tn").shift(1).rolling_mean(3, min_samples=1).over(g).alias("rmean_3"),
        pl.col("tn").shift(1).rolling_mean(12, min_samples=1).over(g).alias("rmean_12"),
        pl.col("tn").shift(1).rolling_std(3, min_samples=2).over(g).alias("rstd_3"),
        (pl.col("tn") > 0).cast(pl.Int32).rolling_sum(12, min_samples=1).shift(1).over(g).alias("ventas_ult12"),
        (pl.col("periodo") % 100).alias("mes"),
    ])
    df = df.with_columns([
        (pl.col("lag_1") / (pl.col("s") + eps)).alias("nivel_relativo"),
        (pl.col("rmean_3") / (pl.col("rmean_12") + eps)).alias("tendencia_3_12"),
        pl.when(pl.col("tn") > 0).then(pl.col("_rn")).otherwise(None).forward_fill().over(g).alias("_uv"),
    ])
    df = df.with_columns((pl.col("_rn") - pl.col("_uv").fill_null(-1)).alias("meses_consec_sin_compra"))

    # agregados del periodo t (presente, conocido al momento de predecir)
    df = df.with_columns([
        pl.col("tn").sum().over(["product_id", "periodo"]).alias("tn_prod"),
        pl.col("tn").sum().over(["cat3", "periodo"]).alias("_tn_cat3"),
        pl.col("tn").sum().over(["customer_id", "periodo"]).alias("tn_cli"),
    ]).with_columns([
        (pl.col("tn_prod") / (pl.col("_tn_cat3") + eps)).alias("share_prod_en_cat3"),
        (pl.col("tn") / (pl.col("tn_prod") + eps)).alias("share_par_en_prod"),
    ])

    # serie del producto: el nivel al que se mide la metrica
    pts = (df.group_by(["product_id", "periodo"])
             .agg(pl.col("tn").sum().alias("_tp"), (pl.col("tn") > 0).sum().alias("_npos"))
             .sort(["product_id", "periodo"])
             .with_columns([
                 pl.col("_tp").rolling_mean(3, min_samples=1).over("product_id").alias("_rm3"),
                 pl.col("_tp").rolling_mean(12, min_samples=1).over("product_id").alias("_rm12"),
                 pl.col("_tp").cum_sum().over("product_id").alias("_ctp"),
                 pl.col("_npos").cum_sum().over("product_id").alias("_cnp"),
             ])
             .with_columns([
                 (pl.col("_rm3") / (pl.col("_rm12") + eps)).alias("tend_prod_3_12"),
                 (pl.col("_ctp") / (pl.col("_cnp") + eps)).alias("nivel_prod_pos"),
             ])
             .select(["product_id", "periodo", "tend_prod_3_12", "nivel_prod_pos"]))
    df = df.join(pts, on=["product_id", "periodo"], how="left")

    # LO UNICO QUE MIRA EL FUTURO
    df = df.with_columns(pl.col("tn").shift(-H).over(g).alias("clase_raw"))

    df = df.with_columns(
        ((pl.col("ventas_ult12").fill_null(0) == 0) & (pl.col("tn") <= eps)).alias("dormido"))

    cols = (["agrupa_id", "customer_id", "product_id", "periodo", "tn", "s", "dormido", "clase_raw"]
            + PARAM["features_base"] + PARAM["features_agregado"])
    return df.select(list(dict.fromkeys(c for c in cols if c in df.columns)))


if PATH_FE.exists():
    print("[features] RESUME"); df_fe = pl.read_parquet(PATH_FE)
else:
    df_fe = _fe(df_pre)
    escribir_atomico(lambda t: df_fe.write_parquet(t), PATH_FE)
print("features:", df_fe.shape,
      "| filas con s=0:", round(100 * (df_fe["s"] <= PARAM["eps"]).mean(), 1), "%",
      "| dormidas:", round(100 * df_fe["dormido"].mean(), 1), "%")

## 5) Test de la ley inviolable (no negociable)

Se recalcula todo el FE sobre la historia **truncada** en un T intermedio. Si alguna
feature de una fila con `periodo <= T` cambia al borrar el futuro, esa feature está
mirando adelante y el notebook se corta acá. Es el test que caza el bug de `rolling` de
z702 (verificado: con el código viejo, este test falla en `rmean_3/rmean_12/rstd_3`).

In [ ]:
def test_causalidad(T=201806, n_prod=25):
    prods = sorted(df_pre["product_id"].unique().to_list())[:n_prod]
    sub = df_pre.filter(pl.col("product_id").is_in(prods))
    full = _fe(sub).filter(pl.col("periodo") <= T).sort(["agrupa_id", "periodo"])
    trunc = _fe(sub.filter(pl.col("periodo") <= T)).sort(["agrupa_id", "periodo"])
    assert full.height == trunc.height, "distinta cantidad de filas"

    malas = []
    for c in [x for x in full.columns if x != "clase_raw"]:
        a, b = full[c], trunc[c]
        if a.dtype in (pl.Utf8, pl.Categorical, pl.Boolean):
            igual = bool((a == b).all())
        else:
            igual = bool(np.allclose(a.fill_null(-9e9).to_numpy(),
                                     b.fill_null(-9e9).to_numpy(), rtol=1e-9, atol=1e-9))
        if not igual:
            malas.append(c)
    if malas:
        raise AssertionError(f"FUGA DE FUTURO en: {malas}")

    dif = int((full["clase_raw"].fill_null(-1).to_numpy()
               != trunc["clase_raw"].fill_null(-1).to_numpy()).sum())
    print(f"[test causalidad] OK — {full.width - 1} columnas idénticas al truncar en {T}. "
          f"clase_raw difiere en {dif} filas, que es lo correcto: es la única que mira adelante.")


test_causalidad()

## 6) Clustering DTW (nivel producto, corte causal)

Se clusteriza la **forma** de la serie agregada de cada producto (1233 series), no la de
los ~700k pares: con 17 % de filas positivas, el DTW entre pares mide sobre todo dónde
caen los ceros. A nivel producto la matriz completa es 1233² y sale en minutos, es el
nivel al que se mide la métrica, y cada par hereda el cluster de su producto.

Ventana común de `cl_ventana` meses terminando en el corte (relleno a izquierda para los
productos jóvenes): todas las series tienen el mismo largo, así que la banda de
Sakoe-Chiba de 3 meses es real. En z702 la banda era `max(3, |Δlargo|)`, que entre series
de largos dispares se abría hasta desactivar la restricción.

**Causalidad**: los clusters se recalculan para cada ancla usando sólo datos `<= ancla`.
El fold de 201712 nunca ve la forma de 2019.

In [ ]:
def _dtw_matriz(series):
    n = len(series)
    if _C_OK:
        M = dtw.distance_matrix_fast(series, window=PARAM["cl_banda"], compact=False)
    else:
        M = np.zeros((n, n))
        for i in range(n):
            for j in range(i + 1, n):
                M[i, j] = dtw.distance(series[i], series[j], window=PARAM["cl_banda"], use_c=False)
    # dtaidistance llena solo el triangulo superior (el resto queda en inf)
    M = np.asarray(M, dtype=np.float64)
    iu = np.triu_indices(n, 1)
    v = np.where(np.isfinite(M[iu]), M[iu], np.nan)
    mx = np.nanmax(v) if v.size and np.isfinite(np.nanmax(v)) else 1.0
    v = np.where(np.isnan(v), mx * 10.0, v)      # banda infactible -> lejos pero finito
    D = np.zeros((n, n)); D[iu] = v
    return D + D.T


def clusters_al_corte(corte: int) -> pl.DataFrame:
    """cluster_id por producto usando SOLO datos <= corte."""
    todos = sorted(df_fe["product_id"].unique().to_list())
    if PARAM["cl_nivel"] == "ninguno":
        return pl.DataFrame({"product_id": todos, "cluster_id": [1] * len(todos)}) \
                 .with_columns(pl.col("cluster_id").cast(pl.Int32))
    path = DIR_OUT / "clusters" / f"corte_{corte}.parquet"
    if path.exists():
        return pl.read_parquet(path)

    W = PARAM["cl_ventana"]
    meses = [periodo_mas_h(corte, -k) for k in range(W - 1, -1, -1)]
    pos = {m: i for i, m in enumerate(meses)}
    ts = (df_fe.filter(pl.col("periodo").is_in(meses))
                .group_by(["product_id", "periodo"]).agg(pl.col("tn").sum().alias("tp")))
    piv = {p: np.zeros(W) for p in todos}
    for p, per, tp in ts.iter_rows():
        piv[p][pos[per]] = tp
    series, vivos = [], []
    for p in todos:
        v = piv[p]
        if v.sum() > PARAM["eps"]:
            series.append(np.ascontiguousarray(v / v.mean(), dtype=np.float64)); vivos.append(p)

    D = _dtw_matriz(series)
    Z = linkage(squareform(D, checks=False), method=PARAM["cl_linkage"])
    cands = []
    for k in PARAM["cl_lista_k"]:
        lab = fcluster(Z, t=k, criterion="maxclust")
        if len(set(lab)) < 2:
            continue
        tam = np.bincount(lab)[1:]
        bal = float(tam[tam > 0].min() / tam.sum())
        sil = float(silhouette_score(D, lab, metric="precomputed"))
        cands.append({"k": k, "sil": sil, "bal": bal, "lab": lab})
        print(f"  corte {corte} k={k}: silhouette={sil:+.4f} balance={bal:.3f}"
              f"{'' if bal >= PARAM['cl_min_balance'] else '  (desbalanceado)'}")
    ok = [c for c in cands if c["bal"] >= PARAM["cl_min_balance"]]
    # si ninguno cumple el balance, se toma el MENOS desbalanceado (no el primero de la lista)
    elegido = max(ok, key=lambda c: c["sil"]) if ok else max(cands, key=lambda c: c["bal"])
    lab = elegido["lab"]
    print(f"  corte {corte} -> k = {len(set(lab))} (silhouette {elegido['sil']:+.4f})")

    out = {p: int(c) for p, c in zip(vivos, lab)}
    mayor = int(np.bincount(lab).argmax())
    df = pl.DataFrame({"product_id": todos, "cluster_id": [out.get(p, mayor) for p in todos]}) \
           .with_columns(pl.col("cluster_id").cast(pl.Int32))
    escribir_atomico(lambda t: df.write_parquet(t), path)
    return df


CORTES = sorted(set(PARAM["anclas_val"] + PARAM["anclas_control"] + [PARAM["periodo_inferencia"]]))
CLUSTERS = {c: clusters_al_corte(c) for c in CORTES}
print({c: CLUSTERS[c].group_by("cluster_id").len().sort("cluster_id")["len"].to_list() for c in CORTES})

## 7) Métrica y multiplicador

WAPE **global** a nivel producto sobre los 780, calculado una sola vez sobre todas las
predicciones ensambladas. `m*` es la mediana ponderada de `y_p/ŷ_p` con pesos `ŷ_p`:
la solución exacta de `argmin_m Σ_p |y_p − m·ŷ_p|`, que es el numerador del WAPE.

In [ ]:
def wape_producto(product_id, y, pred):
    d = pd.DataFrame({"product_id": product_id, "y": y, "p": pred})
    d = d[d["product_id"].isin(PROD_TARGET)]
    g = d.groupby("product_id", as_index=False).agg(y=("y", "sum"), p=("p", "sum"))
    den = g["y"].sum()
    if den <= 0:
        return float("nan"), g
    return float(np.abs(g["y"] - g["p"]).sum() / den), g


def multiplicador_optimo(y, yhat):
    """argmin_m sum |y - m*yhat| = mediana de y/yhat ponderada por yhat."""
    ok = yhat > 0
    if ok.sum() == 0:
        return 1.0
    r, w = y[ok] / yhat[ok], yhat[ok]
    o = np.argsort(r); r, w = r[o], w[o]
    return float(r[np.searchsorted(np.cumsum(w), 0.5 * w.sum())])


def tabla_por_decil(g):
    d = g.copy()
    d["decil"] = pd.qcut(d["p"].rank(method="first"), 10, labels=False) + 1
    t = d.groupby("decil").agg(n=("p", "size"), pred=("p", "sum"), real=("y", "sum"))
    t["ratio"] = t["real"] / t["pred"].replace(0, np.nan)
    return t


def calibracion_afin(y, yhat):
    """argmin_{a,b} sum |y - a - b*yhat| con a >= 0 (regresion cuantil tau=0.5)."""
    def f(ab):
        return float(np.abs(y - max(ab[0], 0.0) - ab[1] * yhat).sum())
    r = minimize(f, x0=[0.0, multiplicador_optimo(y, yhat)], method="Nelder-Mead",
                 options={"maxiter": 3000, "xatol": 1e-6, "fatol": 1e-6})
    return max(float(r.x[0]), 0.0), float(r.x[1])


FEATS = list(PARAM["features_base"])
if PARAM["usar_features_agregado"]:
    FEATS += PARAM["features_agregado"]
if PARAM["usar_cluster_como_feature"] and PARAM["cl_nivel"] != "ninguno":
    FEATS += ["cluster_id"]
CATS = [c for c in PARAM["cols_categoricas"] if c in FEATS]
CAT_IDX = [FEATS.index(c) for c in CATS]

# codigos enteros fijos para las categoricas (LightGBM nativo trabaja con codigos)
COD = {}
for c in CATS:
    if c == "cluster_id":
        vals = sorted({int(v) for d in CLUSTERS.values() for v in d["cluster_id"].to_list()})
    else:
        vals = sorted(x for x in df_fe[c].unique().to_list() if x is not None)
    COD[c] = {v: i for i, v in enumerate(vals)}
print(f"{len(FEATS)} features | categoricas: {CATS}")

## 8) Folds

Para el ancla `st` se entrena con `periodo <= st − 2` y se valida sobre `periodo == st`
(target `st + 2`). El `− 2` replica el hueco real del deploy: parados en 201912, el
último target conocido es el de 201910 y hay que llegar a 202002.

Los `lgb.Dataset` se construyen **una sola vez** por fold (el binneado con `max_bin=1023`
es caro) y se reusan en los 25 trials de Optuna.

In [ ]:
def _submuestrear(d: pd.DataFrame) -> pd.DataFrame:
    tasa = PARAM["submuestreo_dormidos"]
    if tasa >= 1.0:
        return d
    h = ((d["agrupa_id"].to_numpy() * 1000003 + d["periodo"].to_numpy()) % 1000) / 1000.0
    return d[(~d["dormido"].to_numpy()) | (h < tasa)]


def _matriz(d: pd.DataFrame) -> np.ndarray:
    cols = []
    for c in FEATS:
        if c in COD:
            v = d[c].map(COD[c]).to_numpy(dtype="float64")
        else:
            v = pd.to_numeric(d[c], errors="coerce").to_numpy(dtype="float64")
        cols.append(v.astype(np.float32))
    return np.column_stack(cols)


def preparar(corte: int, filtro: pl.Expr, entrenamiento: bool) -> dict:
    d = (df_fe.filter(filtro)
              .join(CLUSTERS[corte], on="product_id", how="left")
              .with_columns(pl.col("cluster_id").fill_null(1).cast(pl.Int32))
              .to_pandas())
    if entrenamiento:
        d = _submuestrear(d)
    w = np.ones(len(d))
    tasa = PARAM["submuestreo_dormidos"]
    if entrenamiento and tasa < 1.0:
        w[d["dormido"].to_numpy()] = 1.0 / tasa
    pack = {"X": _matriz(d), "y": d["clase_raw"].to_numpy(dtype="float64"), "w": w,
            "s": np.maximum(d["s"].to_numpy(dtype="float64"), PISO),
            "pid": d["product_id"].to_numpy(), "n": len(d)}
    del d; gc.collect()
    return pack


def dataset(pack: dict) -> lgb.Dataset:
    """Binnea una sola vez. Con 'offset' el target son toneladas y s va como init_score."""
    if PARAM["modo_escala"] == "offset":
        ds = lgb.Dataset(pack["X"], label=pack["y"], weight=pack["w"],
                         init_score=np.log(pack["s"]), feature_name=FEATS,
                         categorical_feature=CAT_IDX, free_raw_data=True,
                         params={"max_bin": PARAM["max_bin"], "verbosity": -1})
    else:
        ds = lgb.Dataset(pack["X"], label=pack["y"] / pack["s"], weight=pack["w"],
                         feature_name=FEATS, categorical_feature=CAT_IDX, free_raw_data=True,
                         params={"max_bin": PARAM["max_bin"], "verbosity": -1})
    ds.construct()
    pack.pop("X", None); gc.collect()
    return ds


def armar_folds():
    folds = []
    for st in PARAM["anclas_val"] + PARAM["anclas_control"]:
        fin_tr = periodo_mas_h(st, -H)
        p_tr = preparar(st, (pl.col("periodo") <= fin_tr) & pl.col("clase_raw").is_not_null(), True)
        p_ev = preparar(st, (pl.col("periodo") == st) & pl.col("clase_raw").is_not_null(), False)
        if p_tr["n"] == 0 or p_ev["n"] == 0:
            print(f"  fold {st}: vacio, se saltea"); continue
        peso = PARAM["peso_febrero"] if periodo_mas_h(st, H) % 100 == 2 else 1.0
        folds.append({"st": st, "target": periodo_mas_h(st, H), "peso": peso,
                      "ds": dataset(p_tr), "ev": p_ev, "n_tr": p_tr["n"]})
        print(f"  fold ancla={st} target={periodo_mas_h(st, H)} train<={fin_tr} "
              f"filas_tr={p_tr['n']:,} filas_ev={p_ev['n']:,} peso={peso}")
    return folds


FOLDS = armar_folds()
assert FOLDS, "no hay folds"

## 9) Motor: LightGBM Tweedie con el escalado como offset

`init_score = log(max(s, piso))` y `pred = max(s, piso)·exp(f)`. Es el mismo desescalado
multiplicativo que hacía z702 (`pred = s·ŷ`), con dos diferencias que importan: la
pérdida se computa en toneladas (el top 1 % de filas concentra el 74,6 % del tonelaje, y
el WAPE pesa exactamente así) y las filas con `s = 0` ya no arrastran un target
aplastado a cero.

In [ ]:
def _params(base: dict, seed: int) -> dict:
    p = {"objective": "tweedie", "metric": "tweedie", "max_bin": PARAM["max_bin"],
         "num_threads": PARAM["num_threads"], "verbosity": -1, "boosting": "gbdt",
         "force_col_wise": True, "seed": seed, "bagging_seed": seed,
         "feature_fraction_seed": seed, "deterministic": True}
    p.update({k: v for k, v in base.items() if k != "n_estimators"})
    return p


def entrenar(ds: lgb.Dataset, base: dict, seed: int) -> lgb.Booster:
    return lgb.train(_params(base, seed), ds, num_boost_round=int(base["n_estimators"]))


def predecir(bst: lgb.Booster, pack: dict) -> np.ndarray:
    if PARAM["modo_escala"] == "offset":
        f = np.clip(bst.predict(pack["X"], raw_score=True), -30, 30)
        return np.maximum(pack["s"] * np.exp(f), 0.0)
    return np.maximum(bst.predict(pack["X"]) * pack["s"], 0.0)


def _espacio(trial):
    lo, hi = PARAM["tweedie_vp_rango"]
    return {
        "tweedie_variance_power": trial.suggest_float("tweedie_variance_power", lo, hi),
        "num_leaves": trial.suggest_int("num_leaves", 16, 256),
        "max_depth": trial.suggest_int("max_depth", 3, 12),
        "learning_rate": trial.suggest_float("learning_rate", 5e-3, 0.2, log=True),
        "n_estimators": trial.suggest_int("n_estimators", min(200, PARAM["techo_arboles"]),
                                          PARAM["techo_arboles"]),
        "min_child_samples": trial.suggest_int("min_child_samples", 20, 300),
        "feature_fraction": trial.suggest_float("feature_fraction", 0.6, 1.0),
        "bagging_fraction": trial.suggest_float("bagging_fraction", 0.6, 1.0),
        "bagging_freq": 1,
    }


def wape_de_folds(base: dict, seed: int, devolver=False):
    num = den = 0.0; detalle = []
    for fo in FOLDS:
        bst = entrenar(fo["ds"], base, seed)
        pred = predecir(bst, fo["ev"])
        w, g = wape_producto(fo["ev"]["pid"], fo["ev"]["y"], pred)
        if np.isnan(w):
            continue
        num += fo["peso"] * w; den += fo["peso"]
        detalle.append({"fold": fo, "wape": w, "g": g})
    valor = num / den if den else 1e9
    return (valor, detalle) if devolver else valor


PATH_BEST = DIR_OUT / "best_params.json"

if PATH_BEST.exists():
    print("[optuna] RESUME"); BEST = leer_json(PATH_BEST)
elif not _HAY_OPTUNA:
    BEST = {"params": {"tweedie_variance_power": 1.3, "num_leaves": 96, "max_depth": 8,
                       "learning_rate": 0.05, "n_estimators": 600, "min_child_samples": 60,
                       "feature_fraction": 0.85, "bagging_fraction": 0.85, "bagging_freq": 1},
            "wape": None}
    escribir_atomico(lambda t: Path(t).write_text(json.dumps(BEST, indent=2)), PATH_BEST)
else:
    t0 = time.time()

    def objetivo(trial):
        return wape_de_folds(_espacio(trial), PARAM["semillas_ensemble"][0])

    est = optuna.create_study(direction="minimize", study_name="v5", load_if_exists=True,
                              sampler=optuna.samplers.TPESampler(seed=PARAM["semilla"]),
                              storage=f"sqlite:///{DIR_OUT / 'optuna_v5.db'}")
    est.optimize(objetivo, n_trials=PARAM["n_trials"], show_progress_bar=False)
    BEST = {"params": est.best_params, "wape": float(est.best_value)}
    escribir_atomico(lambda t: Path(t).write_text(json.dumps(BEST, indent=2, default=str)), PATH_BEST)
    print(f"optuna: {PARAM['n_trials']} trials en {(time.time() - t0) / 60:.1f} min")

print("WAPE de validacion (ponderado, sin multiplicador):", BEST["wape"])
print("mejores hiperparametros:", json.dumps(BEST["params"], indent=2, default=str))

## 10) Multiplicador: estimación, diagnóstico y encogimiento

Antes de aplicar cualquier escalar hay que mirar la **tabla por decil**: si el cociente
real/predicho es plano, un multiplicador global es lo correcto; si es monótono, el sesgo
es heterogéneo y un escalar mejora un grupo mientras empeora el otro — ahí corresponde
la calibración afín. Con el offset y Tweedie (que ajusta medias, no medianas) `m*`
debería quedar cerca de 1; si se va lejos, es síntoma de otra cosa, no algo para tapar.

In [ ]:
PATH_M = DIR_OUT / "multiplicador.json"

if PATH_M.exists():
    print("[m*] RESUME"); DIAG = leer_json(PATH_M)
else:
    _, detalle = wape_de_folds(BEST["params"], PARAM["semillas_ensemble"][0], devolver=True)
    filas = []
    for det in detalle:
        fo, g, w0 = det["fold"], det["g"], det["wape"]
        y, yh = g["y"].to_numpy(), g["p"].to_numpy()
        m = multiplicador_optimo(y, yh)
        w1 = float(np.abs(y - m * yh).sum() / y.sum())
        a, b = calibracion_afin(y, yh) if PARAM["probar_calibracion_afin"] else (0.0, m)
        w2 = float(np.abs(y - a - b * yh).sum() / y.sum())
        filas.append({"ancla": fo["st"], "target": fo["target"], "peso": fo["peso"], "wape": w0,
                      "m_estrella": m, "wape_con_m": w1, "afin_a": a, "afin_b": b, "wape_afin": w2})
        print(f"\nfold ancla={fo['st']} target={fo['target']}: WAPE={w0:.4f}   "
              f"m*={m:.4f} -> WAPE={w1:.4f}   afin(a={a:.2f}, b={b:.3f}) -> WAPE={w2:.4f}")
        print(tabla_por_decil(g).to_string())

    pesos = np.array([f["peso"] for f in filas])
    ms = np.array([f["m_estrella"] for f in filas])
    m_bar = float(np.average(ms, weights=pesos))
    sigma = float(np.sqrt(np.average((ms - m_bar) ** 2, weights=pesos)))
    m_final = 1.0 + PARAM["lambda_shrink"] * (m_bar - 1.0)
    w_m = float(np.average([f["wape_con_m"] for f in filas], weights=pesos))
    w_a = float(np.average([f["wape_afin"] for f in filas], weights=pesos))
    DIAG = {"folds": filas, "m_bar": m_bar, "sigma": sigma, "lambda": PARAM["lambda_shrink"],
            "m_final": m_final, "wape_con_m": w_m, "wape_afin": w_a,
            "gana_afin": bool(w_a < w_m * (1 - PARAM["margen_afin"])),
            "afin_a": float(np.average([f["afin_a"] for f in filas], weights=pesos)),
            "afin_b": float(np.average([f["afin_b"] for f in filas], weights=pesos))}
    escribir_atomico(lambda t: Path(t).write_text(json.dumps(DIAG, indent=2)), PATH_M)

print(f"\nm* por fold: {[round(f['m_estrella'], 4) for f in DIAG['folds']]}")
print(f"m_barra={DIAG['m_bar']:.4f}  sigma={DIAG['sigma']:.4f}  lambda={DIAG['lambda']}"
      f"  ->  m_final={DIAG['m_final']:.4f}")
print(f"WAPE con m={DIAG['wape_con_m']:.4f} | con afin={DIAG['wape_afin']:.4f} -> "
      f"la afin {'GANA por mas del margen' if DIAG['gana_afin'] else 'no gana lo suficiente'}")

## 11) Modelo final e inferencia

Entrena con todo lo que tiene clase (hasta 201910, target 201912), promedia las semillas
**en toneladas**, predice el ancla 201912 (target 202002) y agrega a nivel producto.

In [ ]:
PATH_PRED = DIR_OUT / "pred_final.parquet"

if PATH_PRED.exists():
    print("[pred] RESUME"); pred_prod = pl.read_parquet(PATH_PRED).to_pandas()
else:
    INF = PARAM["periodo_inferencia"]
    p_tr = preparar(INF, pl.col("clase_raw").is_not_null(), True)
    p_ev = preparar(INF, pl.col("periodo") == INF, False)
    print(f"train final: {p_tr['n']:,} filas (hasta {periodo_mas_h(INF, -H)})  "
          f"| inferencia: {p_ev['n']:,} filas")
    ds_f = dataset(p_tr)
    acum = np.zeros(p_ev["n"])
    for i, sd in enumerate(PARAM["semillas_ensemble"], 1):
        acum += predecir(entrenar(ds_f, BEST["params"], sd), p_ev)
        print(f"  semilla {sd} lista ({i}/{len(PARAM['semillas_ensemble'])})")
    pred_prod = (pd.DataFrame({"product_id": p_ev["pid"],
                               "pred_tn": acum / len(PARAM["semillas_ensemble"])})
                 .groupby("product_id", as_index=False)["pred_tn"].sum())
    escribir_atomico(lambda t: pl.from_pandas(pred_prod).write_parquet(t), PATH_PRED)

print("productos predichos:", len(pred_prod), "| tn total:", round(pred_prod["pred_tn"].sum(), 1))

## 12) Submissions y envío

Se emiten el crudo (m = 1), el `m_final` estimado y `m_final ± σ`, donde σ es la
dispersión medida entre folds. La calibración afín sólo entra si le gana al multiplicador
por más del margen, y con el intercepto acotado: un `a` grande aplicado a los productos
que predicen casi cero los infla sin ninguna evidencia a favor.

In [ ]:
apr = pd.read_csv(DIR_RAW / "product_id_apredecir201912.txt", sep="\t")[["product_id"]]
base_sub = apr.merge(pred_prod.rename(columns={"pred_tn": "tn"}), on="product_id", how="left")
base_sub["tn"] = base_sub["tn"].fillna(0.0)

mf, sg = DIAG["m_final"], DIAG["sigma"]
cands = {"crudo":  ("m1.000", lambda v: v),
         "mfinal": (f"m{mf:.3f}", lambda v: v * mf),
         "mbajo":  (f"m{max(mf - sg, 0.5):.3f}", lambda v: v * max(mf - sg, 0.5)),
         "malto":  (f"m{mf + sg:.3f}", lambda v: v * (mf + sg))}
if DIAG["gana_afin"]:
    pos = base_sub.loc[base_sub["tn"] > 0, "tn"]
    a = min(DIAG["afin_a"], float(np.percentile(pos, 10)) if len(pos) else 0.0)
    b = DIAG["afin_b"]
    cands["afin"] = (f"afin_a{a:.2f}_b{b:.3f}", lambda v: np.where(v > 0, a + b * v, 0.0))

RUTAS = {}
for nombre, (etiqueta, fn) in cands.items():
    sub = base_sub.copy()
    sub["tn"] = np.maximum(fn(sub["tn"].to_numpy()), PARAM["clip_min"])
    ruta = DIR_OUT / f"submission_{nombre}_{etiqueta}.csv"
    escribir_atomico(lambda t, s=sub: s.to_csv(t, index=False), ruta)
    RUTAS[nombre] = (ruta, etiqueta)
    print(f"  {nombre:7s} {etiqueta:18s} filas={len(sub)}  tn_total={sub['tn'].sum():9.1f}  -> {ruta.name}")

In [ ]:
if PARAM["submit"]:
    pr = subprocess.run(["kaggle", "competitions", "list"], capture_output=True, text=True)
    print("kaggle auth:", "OK" if pr.returncode == 0 else f"FALLA -> {(pr.stderr or pr.stdout)[:300]}")
    if pr.returncode == 0:
        ok = 0
        for nombre, (ruta, etiqueta) in RUTAS.items():
            flag = DIR_OUT / "submits" / f"{nombre}.done"
            if flag.exists():
                print(f"  {nombre}: ya enviado, se saltea"); continue
            r = subprocess.run(["kaggle", "competitions", "submit",
                                "-c", PARAM["kaggle_competition"], "-f", str(ruta),
                                "-m", f"z705 offset tweedie {etiqueta}"],
                               capture_output=True, text=True)
            salida = ((r.stdout or "") + (r.stderr or "")).strip()
            print(f"  {nombre}: rc={r.returncode} {salida[:200]}")
            if r.returncode == 0:
                escribir_atomico(lambda t: Path(t).write_text(time.strftime("%Y-%m-%d %H:%M:%S")), flag)
                ok += 1
        print(f"\nenviados OK: {ok}/{len(RUTAS)}")
else:
    print("submit=False -> los CSV quedan en", DIR_OUT)